In [1]:
import requests

# Test 1: Check whether the ChEMBL API responds
status_url = "https://www.ebi.ac.uk/chembl/api/data/status.json"
response = requests.get(status_url, timeout=30)
print("ChEMBL API HTTP status:", response.status_code)
response.raise_for_status()

status_data = response.json()
print("API status response:")
print(status_data)

# Test 2: Request just five approved-drug records
drugs_url = "https://www.ebi.ac.uk/chembl/api/data/molecule.json"
drug_response = requests.get(
    drugs_url,
    params={
        "max_phase": 4,
        "limit": 5
    },
    timeout=30
)

print("\nDrug query HTTP status:", drug_response.status_code)
drug_response.raise_for_status()

drug_data = drug_response.json()
molecules = drug_data.get("molecules", [])
print("Number of returned records:", len(molecules))

for molecule in molecules:
    print({
        "ChEMBL_ID": molecule.get("molecule_chembl_id"),
        "Name": molecule.get("pref_name"),
        "Maximum_Clinical_Phase": molecule.get("max_phase")
    })


ChEMBL API HTTP status: 200
API status response:
{'activities': 24527044, 'chembl_db_version': 'ChEMBL_37', 'chembl_release_date': '2026-05-01', 'compound_records': 3824604, 'disinct_compounds': 2921148, 'publications': 101100, 'status': 'UP', 'targets': 18552}



Drug query HTTP status: 200
Number of returned records: 5
{'ChEMBL_ID': 'CHEMBL2', 'Name': 'PRAZOSIN', 'Maximum_Clinical_Phase': '4.0'}
{'ChEMBL_ID': 'CHEMBL3', 'Name': 'NICOTINE', 'Maximum_Clinical_Phase': '4.0'}
{'ChEMBL_ID': 'CHEMBL4', 'Name': 'OFLOXACIN', 'Maximum_Clinical_Phase': '4.0'}
{'ChEMBL_ID': 'CHEMBL5', 'Name': 'NALIDIXIC ACID', 'Maximum_Clinical_Phase': '4.0'}
{'ChEMBL_ID': 'CHEMBL6', 'Name': 'INDOMETHACIN', 'Maximum_Clinical_Phase': '4.0'}


In [2]:
import time
import requests
import pandas as pd
from IPython.display import display

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

# Small pilot panel of genes relevant to GBM
test_genes = [
    "EGFR", "TP53", "PTEN", "PIK3CA",
    "CDK4", "MDM2", "BRAF", "MGMT"
]

session = requests.Session()
session.headers.update({
    "User-Agent": "PersonalizedGBMPrioritization/1.0"
})

def get_json(endpoint, params=None):
    """Request a ChEMBL JSON endpoint and check for HTTP errors."""
    response = session.get(
        f"{BASE_URL}/{endpoint.lstrip('/')}",
        params=params,
        timeout=30
    )
    response.raise_for_status()
    return response.json()

target_rows = []
candidate_rows = []

# Search ChEMBL and map each gene to an exact human gene symbol
for gene in test_genes:
    print(f"Searching ChEMBL targets for {gene}...")
    search_data = get_json(
        "target/search.json",
        params={"q": gene, "limit": 50}
    )
    candidates = search_data.get("targets", [])
    matched = False
    for candidate in candidates:
        if candidate.get("organism") != "Homo sapiens":
            continue
        if candidate.get("target_type") != "SINGLE PROTEIN":
            continue
        target_id = candidate.get("target_chembl_id")
        if not target_id:
            continue
        candidate_rows.append({
            "Queried_Gene": gene,
            "Target_ChEMBL_ID": target_id,
            "Target_Name": candidate.get("pref_name"),
            "Organism": candidate.get("organism"),
            "Target_Type": candidate.get("target_type")
        })
        # Target search may not include all component details.
        target_details = candidate
        components = candidate.get("target_components") or []
        if not components:
            target_details = get_json(f"target/{target_id}.json")
            components = target_details.get("target_components") or []
        
        for component in components:
            synonyms = component.get("target_component_synonyms") or []
            for syn in synonyms:
                if syn.get("component_synonym", "").strip().upper() == gene.upper():
                    target_rows.append({
                        "Gene_Symbol": gene,
                        "Target_ChEMBL_ID": target_id,
                        "Target_Name": candidate.get("pref_name"),
                        "Target_Type": candidate.get("target_type"),
                        "Organism": candidate.get("organism")
                    })
                    matched = True
                    break
            if matched:
                break
        if matched:
            break

targets_df = pd.DataFrame(target_rows)

print("\n--- Exact Human Target Mappings ---")
mapped_genes = targets_df["Gene_Symbol"].tolist() if not targets_df.empty else []
print("Genes with matches:", mapped_genes)
unmapped_genes = [g for g in test_genes if g not in mapped_genes]
print("Genes without exact matches:", unmapped_genes if unmapped_genes else "None")

if not targets_df.empty:
    display(targets_df)
else:
    print("No exact matches found in this query; inspect the API response.")

# Pilot activity retrieval for EGFR only
egfr_targets = targets_df[targets_df["Gene_Symbol"] == "EGFR"]

if egfr_targets.empty:
    print("\nEGFR activity test skipped: no exact EGFR target was mapped.")
else:
    egfr_target_id = egfr_targets.iloc[0]["Target_ChEMBL_ID"]
    activity_response = session.get(
        f"{BASE_URL}/activity.json",
        params={
            "target_chembl_id": egfr_target_id,
            "standard_type": "IC50",
            "limit": 5
        },
        timeout=30
    )
    activity_response.raise_for_status()
    activities = activity_response.json().get("activities", [])
    activity_columns = [
        "molecule_chembl_id",
        "target_chembl_id",
        "standard_type",
        "standard_relation",
        "standard_value",
        "standard_units",
        "pchembl_value",
        "assay_type",
        "confidence_score"
    ]
    activities_df = pd.DataFrame(activities)
    print(f"\nEGFR target ID: {egfr_target_id}")
    print("Activity endpoint HTTP status:", activity_response.status_code)
    print("Returned activity records:", len(activities))
    available_columns = [
        col for col in activity_columns if col in activities_df.columns
    ]
    if activities_df.empty:
        print("No IC50 activity records returned in this sample.")
    else:
        display(activities_df[available_columns])


Searching ChEMBL targets for EGFR...


Searching ChEMBL targets for TP53...


Searching ChEMBL targets for PTEN...


Searching ChEMBL targets for PIK3CA...


Searching ChEMBL targets for CDK4...


...

Searching ChEMBL targets for MGMT...



--- Exact Human Target Mappings ---
Genes with matches: ['EGFR', 'TP53', 'PTEN', 'PIK3CA', 'CDK4', 'MDM2', 'BRAF', 'MGMT']
Genes without exact matches: None


,Gene_Symbol,Target_ChEMBL_ID,Target_Name,Target_Type,Organism
0,EGFR,CHEMBL203,Epidermal growth factor receptor,SINGLE PROTEIN,Homo sapiens
1,TP53,CHEMBL4096,Cellular tumor antigen p53,SINGLE PROTEIN,Homo sapiens
2,PTEN,CHEMBL2052032,"Phosphatidylinositol 3,4,5-trisphosphate 3-pho...",SINGLE PROTEIN,Homo sapiens
3,PIK3CA,CHEMBL4005,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",SINGLE PROTEIN,Homo sapiens
4,CDK4,CHEMBL331,Cyclin-dependent kinase 4,SINGLE PROTEIN,Homo sapiens
5,MDM2,CHEMBL5023,E3 ubiquitin-protein ligase Mdm2,SINGLE PROTEIN,Homo sapiens
6,BRAF,CHEMBL5145,Serine/threonine-protein kinase B-raf,SINGLE PROTEIN,Homo sapiens
7,MGMT,CHEMBL2864,Methylated-DNA--protein-cysteine methyltransfe...,SINGLE PROTEIN,Homo sapiens



EGFR target ID: CHEMBL203
Activity endpoint HTTP status: 200
Returned activity records: 5


,molecule_chembl_id,target_chembl_id,standard_type,standard_relation,standard_value,standard_units,pchembl_value,assay_type
0,CHEMBL68920,CHEMBL203,IC50,=,41.0,nM,7.39,B
1,CHEMBL68920,CHEMBL203,IC50,=,300.0,nM,6.52,F
2,CHEMBL68920,CHEMBL203,IC50,=,7820.0,nM,5.11,F
3,CHEMBL69960,CHEMBL203,IC50,=,170.0,nM,6.77,B
4,CHEMBL69960,CHEMBL203,IC50,=,40.0,nM,7.40,F


In [3]:
import time

session.headers.update({
    "User-Agent": "PersonalizedGBMPrioritization/1.0"
})

target_ids = dict(
    zip(targets_df["Gene_Symbol"], targets_df["Target_ChEMBL_ID"])
)

coverage_rows = []

for gene, target_id in target_ids.items():
    try:
        response = session.get(
            f"{BASE_URL}/activity.json",
            params={
                "target_chembl_id": target_id,
                "standard_type": "IC50",
                "limit": 1,
            },
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        page_meta = data.get("page_meta", {})
        activities = data.get("activities", [])
        coverage_rows.append({
            "Gene_Symbol": gene,
            "Target_ChEMBL_ID": target_id,
            "Reported_IC50_Records": page_meta.get("total_count"),
            "Example_Record_Returned": len(activities),
            "Query_Status": "Success",
        })
    except (requests.RequestException, ValueError) as error:
        coverage_rows.append({
            "Gene_Symbol": gene,
            "Target_ChEMBL_ID": target_id,
            "Reported_IC50_Records": None,
            "Example_Record_Returned": 0,
            "Query_Status": f"Error: {error}",
        })
    time.sleep(0.2)

coverage_df = pd.DataFrame(coverage_rows)

print("ChEMBL IC50 activity coverage")
display(coverage_df)

print(
    "\nSuccessful target queries:",
    (coverage_df["Query_Status"] == "Success").sum(),
    "of",
    len(target_ids),
)


ChEMBL IC50 activity coverage


,Gene_Symbol,Target_ChEMBL_ID,Reported_IC50_Records,Example_Record_Returned,Query_Status
0,EGFR,CHEMBL203,26600,1,Success
1,TP53,CHEMBL4096,29,1,Success
2,PTEN,CHEMBL2052032,3,1,Success
3,PIK3CA,CHEMBL4005,10031,1,Success
4,CDK4,CHEMBL331,1787,1,Success
5,MDM2,CHEMBL5023,5820,1,Success
6,BRAF,CHEMBL5145,11017,1,Success
7,MGMT,CHEMBL2864,147,1,Success



Successful target queries: 8 of 8


In [4]:
# Step 47.4A — Retrieve a small activity sample
sample_records = []
coverage_records = []
query_errors = []

for gene, target_id in target_ids.items():
    try:
        response = session.get(
            f"{BASE_URL}/activity.json",
            params={
                "target_chembl_id": target_id,
                "standard_type": "IC50",
                "limit": 10,
            },
            timeout=30,
        )
        response.raise_for_status()
        result = response.json()
        coverage_records.append({
            "Gene_Symbol": gene,
            "Target_ChEMBL_ID": target_id,
            "Reported_IC50_Records": result.get(
                "page_meta", {}
            ).get("total_count"),
            "Sampled_Records": len(result.get("activities", [])),
        })
        for record in result.get("activities", []):
            sample_records.append({
                "Gene_Symbol": gene,
                "Target_ChEMBL_ID": target_id,
                "Molecule_ChEMBL_ID": record.get("molecule_chembl_id"),
                "Assay_ChEMBL_ID": record.get("assay_chembl_id"),
                "Standard_Type": record.get("standard_type"),
                "Standard_Relation": record.get("standard_relation"),
                "Standard_Value": record.get("standard_value"),
                "Standard_Units": record.get("standard_units"),
                "pChEMBL_Value": record.get("pchembl_value"),
                "Assay_Type": record.get("assay_type"),
                "Confidence_Score": record.get("confidence_score"),
                "Data_Validity_Comment": record.get(
                    "data_validity_comment"
                ),
            })
    except (requests.RequestException, ValueError) as error:
        query_errors.append({
            "Gene_Symbol": gene,
            "Target_ChEMBL_ID": target_id,
            "Error": str(error),
        })
    time.sleep(0.2)

sample_df = pd.DataFrame(sample_records)
coverage_df = pd.DataFrame(coverage_records)
errors_df = pd.DataFrame(query_errors)

print("\n=== Retrieval coverage ===")
display(coverage_df)
print("\nTotal activity records sampled:", len(sample_df))
print("Failed target queries:", len(errors_df))
if not errors_df.empty:
    display(errors_df)



=== Retrieval coverage ===


,Gene_Symbol,Target_ChEMBL_ID,Reported_IC50_Records,Sampled_Records
0,EGFR,CHEMBL203,26600,10
1,TP53,CHEMBL4096,29,10
2,PTEN,CHEMBL2052032,3,3
3,PIK3CA,CHEMBL4005,10031,10
4,CDK4,CHEMBL331,1787,10
5,MDM2,CHEMBL5023,5820,10
6,BRAF,CHEMBL5145,11017,10
7,MGMT,CHEMBL2864,147,10



Total activity records sampled: 73
Failed target queries: 0


In [5]:
# Step 47.4B — Audit values, units, IDs and missingness
if sample_df.empty:
    print("No activity records were retrieved. Check the query errors.")
else:
    audit_df = sample_df.copy()
    # Normalize blank text fields to missing values.
    text_fields = [
        "Molecule_ChEMBL_ID",
        "Assay_ChEMBL_ID",
        "Standard_Type",
        "Standard_Relation",
        "Standard_Units",
        "Assay_Type",
        "Data_Validity_Comment",
    ]
    for column in text_fields:
        audit_df[column] = audit_df[column].replace(
            r"^\s*$", pd.NA, regex=True
        )
    # Convert measurements to numeric values for quality checks.
    audit_df["Standard_Value_Numeric"] = pd.to_numeric(
        audit_df["Standard_Value"], errors="coerce"
    )
    audit_df["pChEMBL_Numeric"] = pd.to_numeric(
        audit_df["pChEMBL_Value"], errors="coerce"
    )
    audit_df["Confidence_Numeric"] = pd.to_numeric(
        audit_df["Confidence_Score"], errors="coerce"
    )
    audit_df["Valid_Positive_Value"] = (
        audit_df["Standard_Value_Numeric"] > 0
    )

    print("=== Overall Missingness ===")
    missingness = pd.DataFrame({
        "Missing_Count": audit_df.isna().sum(),
        "Missing_Percent": (audit_df.isna().mean() * 100).round(1)
    })
    display(missingness)

    print("\n=== Data Quality Summary ===")
    quality_summary = pd.DataFrame([{
        "Total_Sampled_Records": len(audit_df),
        "Valid_Positive_Numeric": int(audit_df["Valid_Positive_Value"].sum()),
        "Valid_Positive_Pct": f"{(audit_df['Valid_Positive_Value'].mean() * 100):.1f}%",
        "Has_pChEMBL": int(audit_df["pChEMBL_Numeric"].notna().sum()),
        "Has_Confidence_Score": int(audit_df["Confidence_Numeric"].notna().sum()),
        "Standard_Units": dict(audit_df["Standard_Units"].value_counts()),
    }])
    display(quality_summary)


=== Overall Missingness ===


,Missing_Count,Missing_Percent
Gene_Symbol,0,0.0
Target_ChEMBL_ID,0,0.0
Molecule_ChEMBL_ID,0,0.0
Assay_ChEMBL_ID,0,0.0
Standard_Type,0,0.0
Standard_Relation,0,0.0
Standard_Value,0,0.0
Standard_Units,0,0.0
pChEMBL_Value,7,9.6
Assay_Type,0,0.0



=== Data Quality Summary ===


,Total_Sampled_Records,Valid_Positive_Numeric,Valid_Positive_Pct,Has_pChEMBL,Has_Confidence_Score,Standard_Units
0,73,73,100.0%,66,0,"{'nM': 72, 'ug.mL-1': 1}"


In [6]:
# Step 47.4C — Inspect representative records
if sample_df.empty:
    print("No records available to display.")
else:
    columns_to_show = [
        "Gene_Symbol",
        "Molecule_ChEMBL_ID",
        "Assay_ChEMBL_ID",
        "Standard_Type",
        "Standard_Relation",
        "Standard_Value",
        "Standard_Units",
        "pChEMBL_Value",
        "Assay_Type",
        "Confidence_Score",
        "Data_Validity_Comment",
    ]
    print("=== First three sampled records per gene ===")
    display(
        sample_df.groupby(
            "Gene_Symbol", group_keys=False
        ).head(3)[columns_to_show]
    )


=== First three sampled records per gene ===


,Gene_Symbol,Molecule_ChEMBL_ID,Assay_ChEMBL_ID,Standard_Type,Standard_Relation,Standard_Value,Standard_Units,pChEMBL_Value,Assay_Type,Confidence_Score,Data_Validity_Comment
0,EGFR,CHEMBL68920,CHEMBL674637,IC50,=,41.0,nM,7.39,B,None,None
1,EGFR,CHEMBL68920,CHEMBL621151,IC50,=,300.0,nM,6.52,F,None,None
2,EGFR,CHEMBL68920,CHEMBL615325,IC50,=,7820.0,nM,5.11,F,None,None
10,TP53,CHEMBL191334,CHEMBL1052162,IC50,=,1390.0,nM,5.86,B,None,None
11,TP53,CHEMBL583413,CHEMBL1052162,IC50,=,3000.0,nM,5.52,B,None,None
12,TP53,CHEMBL584512,CHEMBL1052162,IC50,=,11400.0,nM,4.94,B,None,None
20,PTEN,CHEMBL2000194,CHEMBL2061290,IC50,=,10000.0,nM,5.00,B,None,None
21,PTEN,CHEMBL2057662,CHEMBL2061290,IC50,=,8870.0,nM,5.05,B,None,None
22,PTEN,CHEMBL301982,CHEMBL4372450,IC50,=,13300.0,nM,4.88,B,None,None
23,PIK3CA,CHEMBL435507,CHEMBL829496,IC50,=,2400.0,nM,5.62,B,None,None


In [7]:
# Step 47.5 — Apply preliminary quality flags
import numpy as np

curated_pilot = sample_df.copy()

curated_pilot["Standard_Value_Numeric"] = pd.to_numeric(
    curated_pilot["Standard_Value"], errors="coerce"
)
curated_pilot["Exact_Relation"] = curated_pilot["Standard_Relation"] == "="
curated_pilot["Units_are_nM"] = curated_pilot["Standard_Units"] == "nM"
curated_pilot["Valid_Positive"] = curated_pilot["Standard_Value_Numeric"] > 0

eligible = (
    curated_pilot["Exact_Relation"]
    & curated_pilot["Units_are_nM"]
    & curated_pilot["Valid_Positive"]
)

curated_pilot["Strict_IC50_candidate"] = eligible

# Derived pIC50 = -log10(IC50 in M) = 9 - log10(IC50 in nM)
curated_pilot["Derived_pIC50"] = np.where(
    curated_pilot["Units_are_nM"] & (curated_pilot["Standard_Value_Numeric"] > 0),
    (9.0 - np.log10(curated_pilot["Standard_Value_Numeric"])).round(2),
    np.nan
)

pilot_summary = curated_pilot.groupby("Gene_Symbol").agg(
    Total_Records=("Molecule_ChEMBL_ID", "count"),
    Strict_Candidates=("Strict_IC50_candidate", "sum"),
    Non_Exact_Relation=("Exact_Relation", lambda x: (~x).sum()),
    Non_nM_Units=(
        "Units_are_nM",
        lambda x: (~x).sum()
    ),
    Missing_pChEMBL=(
        "pChEMBL_Value",
        lambda x: x.isna().sum()
    ),
).reset_index()

print("=== Preliminary quality flags by target ===")
display(pilot_summary)

print("\nTotal records:", len(curated_pilot))
print(
    "Strict quantitative candidates:",
    int(curated_pilot["Strict_IC50_candidate"].sum())
)

# Show records that need review, without removing them.
exceptions = curated_pilot[
    ~curated_pilot["Strict_IC50_candidate"]
][[
    "Gene_Symbol",
    "Molecule_ChEMBL_ID",
    "Assay_ChEMBL_ID",
    "Standard_Relation",
    "Standard_Value",
    "Standard_Units",
    "pChEMBL_Value",
    "Derived_pIC50",
]]

print("\n=== Records excluded from the strict pilot subset ===")
display(exceptions)

# Compare supplied and derived pChEMBL values where both exist.
comparison = curated_pilot[
    eligible & curated_pilot["pChEMBL_Value"].notna()
].copy()

comparison["pChEMBL_numeric"] = pd.to_numeric(
    comparison["pChEMBL_Value"], errors="coerce"
)
comparison["Difference_abs"] = (
    comparison["Derived_pIC50"] -
    comparison["pChEMBL_numeric"]
).abs()

print("\n=== Supplied versus derived potency values ===")
if comparison.empty:
    print("No records available for comparison.")
else:
    display(comparison[[
        "Gene_Symbol",
        "Molecule_ChEMBL_ID",
        "Standard_Value",
        "Standard_Units",
        "pChEMBL_Value",
        "Derived_pIC50",
        "Difference_abs",
    ]].head(15))


=== Preliminary quality flags by target ===


,Gene_Symbol,Total_Records,Strict_Candidates,Non_Exact_Relation,Non_nM_Units,Missing_pChEMBL
0,BRAF,10,9,1,0,1
1,CDK4,10,10,0,0,0
2,EGFR,10,10,0,0,2
3,MDM2,10,9,0,1,1
4,MGMT,10,10,0,0,0
5,PIK3CA,10,7,3,0,3
6,PTEN,3,3,0,0,0
7,TP53,10,10,0,0,0



Total records: 73
Strict quantitative candidates: 68

=== Records excluded from the strict pilot subset ===


,Gene_Symbol,Molecule_ChEMBL_ID,Assay_ChEMBL_ID,Standard_Relation,Standard_Value,Standard_Units,pChEMBL_Value,Derived_pIC50
26,PIK3CA,CHEMBL179242,CHEMBL829496,>,100000.0,nM,None,4.0
27,PIK3CA,CHEMBL176064,CHEMBL828046,>,100000.0,nM,None,4.0
28,PIK3CA,CHEMBL175944,CHEMBL828046,>,100000.0,nM,None,4.0
49,MDM2,CHEMBL199341,CHEMBL869434,=,50.0,ug.mL-1,None,NaN
53,BRAF,CHEMBL185238,CHEMBL829087,>,16000.0,nM,None,4.8



=== Supplied versus derived potency values ===


,Gene_Symbol,Molecule_ChEMBL_ID,Standard_Value,Standard_Units,pChEMBL_Value,Derived_pIC50,Difference_abs
0,EGFR,CHEMBL68920,41.0,nM,7.39,7.39,0.0
1,EGFR,CHEMBL68920,300.0,nM,6.52,6.52,0.0
2,EGFR,CHEMBL68920,7820.0,nM,5.11,5.11,0.0
3,EGFR,CHEMBL69960,170.0,nM,6.77,6.77,0.0
4,EGFR,CHEMBL69960,40.0,nM,7.40,7.40,0.0
5,EGFR,CHEMBL69960,440.0,nM,6.36,6.36,0.0
6,EGFR,CHEMBL137635,9300.0,nM,5.03,5.03,0.0
9,EGFR,CHEMBL77085,96000.0,nM,4.02,4.02,0.0
10,TP53,CHEMBL191334,1390.0,nM,5.86,5.86,0.0
11,TP53,CHEMBL583413,3000.0,nM,5.52,5.52,0.0


In [8]:
# Step 47.5 correction — Verify exact values and censored bounds
corrected = sample_df.copy()

corrected["Value_nM_numeric"] = pd.to_numeric(
    corrected["Standard_Value"], errors="coerce"
)
units = corrected["Standard_Units"].str.strip().str.lower()
relation = corrected["Standard_Relation"].str.strip()

positive = corrected["Value_nM_numeric"] > 0
is_nM = units.eq("nm")

# Exact pIC50 values are calculated only for exact, positive nM measurements.
exact_mask = (
    positive
    & is_nM
    & relation.eq("=")
    & corrected["Molecule_ChEMBL_ID"].notna()
    & corrected["Assay_ChEMBL_ID"].notna()
)
corrected["Derived_pIC50"] = np.nan
corrected.loc[exact_mask, "Derived_pIC50"] = (
    9 - np.log10(corrected.loc[exact_mask, "Value_nM_numeric"])
)

# Censored values are recorded as bounds, not exact pIC50 measurements.
censored_mask = positive & is_nM & relation.isin([">", "<"])
corrected["pIC50_Bound_Value"] = np.nan
corrected["pIC50_Bound_Relation"] = pd.NA
corrected.loc[censored_mask, "pIC50_Bound_Value"] = (
    9 - np.log10(corrected.loc[censored_mask, "Value_nM_numeric"])
)

# Because pIC50 reverses the direction of IC50 inequalities:
# IC50 > x implies pIC50 < calculated threshold.
# IC50 < x implies pIC50 > calculated threshold.
corrected.loc[relation.eq(">") & censored_mask, "pIC50_Bound_Relation"] = "<"
corrected.loc[relation.eq("<") & censored_mask, "pIC50_Bound_Relation"] = ">"

print("Exact quantitative candidates:", int(exact_mask.sum()))
print("Censored nM measurements:", int(censored_mask.sum()))

print("\n=== Censored measurements: bounds only ===")
display(corrected.loc[censored_mask, [
    "Gene_Symbol",
    "Molecule_ChEMBL_ID",
    "Standard_Relation",
    "Standard_Value",
    "Standard_Units",
    "Derived_pIC50",
    "pIC50_Bound_Relation",
    "pIC50_Bound_Value",
]])

print("\n=== Non-nM measurements requiring separate handling ===")
display(corrected.loc[positive & ~is_nM, [
    "Gene_Symbol",
    "Molecule_ChEMBL_ID",
    "Standard_Relation",
    "Standard_Value",
    "Standard_Units",
    "Derived_pIC50",
]])

# Check agreement with reported pChEMBL on exact measurements.
reported = pd.to_numeric(corrected["pChEMBL_Value"], errors="coerce")
compare_mask = exact_mask & reported.notna()

if compare_mask.any():
    differences = (
        corrected.loc[compare_mask, "Derived_pIC50"] - reported.loc[compare_mask]
    ).abs()
    print("\nExact measurements compared with reported pChEMBL:")
    print("Compared records:", int(compare_mask.sum()))
    print("Maximum absolute difference:", round(float(differences.max()), 6))
else:
    print("\nNo exact records with reported pChEMBL values to compare.")


Exact quantitative candidates: 68
Censored nM measurements: 4

=== Censored measurements: bounds only ===


,Gene_Symbol,Molecule_ChEMBL_ID,Standard_Relation,Standard_Value,Standard_Units,Derived_pIC50,pIC50_Bound_Relation,pIC50_Bound_Value
26,PIK3CA,CHEMBL179242,>,100000.0,nM,NaN,<,4.00000
27,PIK3CA,CHEMBL176064,>,100000.0,nM,NaN,<,4.00000
28,PIK3CA,CHEMBL175944,>,100000.0,nM,NaN,<,4.00000
53,BRAF,CHEMBL185238,>,16000.0,nM,NaN,<,4.79588



=== Non-nM measurements requiring separate handling ===


,Gene_Symbol,Molecule_ChEMBL_ID,Standard_Relation,Standard_Value,Standard_Units,Derived_pIC50
49,MDM2,CHEMBL199341,=,50.0,ug.mL-1,NaN



Exact measurements compared with reported pChEMBL:
Compared records: 66
Maximum absolute difference: 0.00515
